In [1]:
%pip install fastparquet python-snappy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pymongo
import pandas as pd
import json
import os
from tqdm import tqdm
from fastparquet import write

# 1. Load checkpoint để lấy danh sách file "sạch"
with open('checkpoint_v2.json', 'r', encoding='utf-8') as f:
    checkpoint = json.load(f)
# Lấy tên file (loại bỏ đường dẫn tuyệt đối của Windows)
valid_files = [os.path.basename(f) for f in checkpoint['completed_files']]

# 2. Kết nối Mongo
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["telegram_scam"]
col = db["messages"]

# 3. Truy vấn chỉ lấy những file trong checkpoint
query = {"source_file": {"$in": valid_files}}
# Chỉ lấy các trường quan trọng để nhẹ file
projection = {
    "_id": 0, "cleaned_text": 1, "label": 1, 
    "has_url": 1, "n_subscribers": 1, "verified": 1
}

cursor = col.find(query, projection).batch_size(50000)

# 4. Xuất theo từng cụm (Sharding) để không treo máy
data = []
part = 1
chunk_size = 1000000

for doc in tqdm(cursor, total=31205696):
    # Loại bỏ các trường ID của MongoDB nếu có để tránh lỗi kiểu dữ liệu
    if '_id' in doc:
        doc['_id'] = str(doc['_id'])
    
    data.append(doc)
    
    if len(data) >= chunk_size:
        df = pd.DataFrame(data)
        
        # Ép kiểu dữ liệu về string cho các cột Object để an toàn tuyệt đối
        for col in df.columns:
            if df[col].dtype == 'object':
                df[col] = df[col].astype(str)
        
        # Ghi file dùng fastparquet trực tiếp
        write(f'data_final_part_{part}.parquet', df, compression='SNAPPY')
        
        data = []
        part += 1

if data:
    df = pd.DataFrame(data)
    write(f'data_final_part_{part}.parquet', df, compression='SNAPPY')

34245696it [17:54, 31864.28it/s]                               


In [4]:
import pandas as pd
import glob

files = glob.glob("data_final_part_*.parquet")
total_rows = 0
for f in files:
    df = pd.read_parquet(f, columns=['label']) # Chỉ load cột label cho nhẹ
    total_rows += len(df)

print(f"Tổng số bản ghi thực tế đã xuất: {total_rows}")
print(f"Số lượng dự kiến theo checkpoint: 31,205,696")

Tổng số bản ghi thực tế đã xuất: 34245696
Số lượng dự kiến theo checkpoint: 31,205,696
